<a href="https://github.com/N3iKos/segsmaker-prallel">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

*   get your civitai api key from [here](https://civitai.com/user/account)


In [ ]:
# @title 🖥️ **WebUI Installer** {"display-mode":"form"}
# @markdown ### Step 1 — Pick your WebUI
Webui = 'Forge' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown ---
# @markdown ### Step 2 — API Keys
# @markdown > Get Civitai key -> https://civitai.com/user/account
Civitai_Key = '' # @param {type:"string", placeholder:"Your Civitai API Key (required)"}
# @markdown > Get HF token -> https://huggingface.co/settings/tokens
HF_Read_Token = '' # @param {type:"string", placeholder:"Your Huggingface READ Token (optional)"}
# @markdown ---
# @markdown ### Step 3 — Google Drive *(optional — for persistent storage)*
# @markdown > If **Yes**, models are saved to `MyDrive/Segsmaker/` and survive session resets.
Mount_GDrive = 'No' # @param ["Yes", "No"]
# @markdown ---
# @markdown ### Step 4 — Fast Setup Downloads
Parallel_Setup_Download = True # @param {type:"boolean"}
Setup_Max_Workers = 6 # @param {type:"slider", min:1, max:8, step:1}
Setup_Aria_Connections = 16 # @param {type:"slider", min:1, max:16, step:1}
Setup_Aria_Split = 16 # @param {type:"slider", min:1, max:16, step:1}
Setup_Min_Split_Size = '1M' # @param ["1M", "2M", "4M", "8M", "16M"]
Setup_Skip_Completed_Files = True # @param {type:"boolean"}
Setup_Fallback_To_Wget = True # @param {type:"boolean"}

import subprocess, sys
from pathlib import Path

if Mount_GDrive == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

_setup_py = '/content/setup.py'
_url = 'https://raw.githubusercontent.com/N3iKos/segsmaker-prallel/main/script/KC/setup.py'
_result = subprocess.run(['curl', '-fLo', _setup_py, _url], capture_output=True, text=True)
if _result.returncode != 0:
    print(f'Setup script download failed:\n{_result.stderr}')
    sys.exit(1)

print('[INFO] Setup script downloaded. Running installer...')
get_ipython().run_line_magic(
    'run',
    f'{_setup_py} --webui="{Webui}" --civitai_key="{Civitai_Key}" --hf_read_token="{HF_Read_Token}" '
    f'--parallel_downloads="{Parallel_Setup_Download}" --max_parallel_downloads="{Setup_Max_Workers}" '
    f'--aria_connections="{Setup_Aria_Connections}" --aria_split="{Setup_Aria_Split}" '
    f'--min_split_size="{Setup_Min_Split_Size}" --skip_completed_files="{Setup_Skip_Completed_Files}" '
    f'--fallback_to_wget="{Setup_Fallback_To_Wget}"'
)

if Mount_GDrive == 'Yes':
    d = Path('/content/drive/MyDrive/Segsmaker')
    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        f = d / n
        f.mkdir(parents=True, exist_ok=True)
        s = p / f'drive-{n}'
        if not s.exists():
            s.symlink_to(f, target_is_directory=True)

    get_ipython().system(f'rm -rf {WebUI_Output}')
    o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    o.mkdir(parents=True, exist_ok=True)
    WebUI_Output.symlink_to(o, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        wc = WebUI / 'cache'
        get_ipython().system(f'rm -rf {wc}')
        c = d / 'cache'
        c.mkdir(parents=True, exist_ok=True)
        wc.symlink_to(c, target_is_directory=True)


## 📥 Model Downloader


In [ ]:
# @title 📥 **Model Downloader** — 5 Checkpoint + 5 LoRA + 1 VAE {"display-mode":"form"}
# @markdown ### 🗃️ Checkpoints
# @markdown > Examples:
# @markdown > - `https://civitai.com/api/download/models/357609`
# @markdown > - `https://civitai.com/models/133005`
# @markdown > - `https://huggingface.co/pantat88/back_up/resolve/main/bigblu25dmix25DStyle_v10.safetensors`
Checkpoint_1 = 'https://huggingface.co/pantat88/back_up/resolve/main/bigblu25dmix25DStyle_v10.safetensors' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_2 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_3 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_4 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_5 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### 🎨 LoRA
# @markdown > Examples:
# @markdown > - `https://civitai.com/models/122359` <- Detail Tweaker XL
# @markdown > - `https://civitai.com/models/669571` <- Pony Add More Details
# @markdown > - `https://huggingface.co/Linaqruf/style-enhancer-xl-lora/resolve/main/style-enhancer-xl.safetensors`
Lora_1 = 'https://civitai.com/models/122359/detail-tweaker-xl' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_2 = 'https://civitai.com/models/669571/pony-add-more-details details-add-more-pony.safetensors' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_3 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_4 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_5 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### 🎛️ VAE
VAE_URL = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### ⚡ Speed Options
download_mode = 'parallel' # @param ["parallel", "sequence"]
parallel_workers = 6 # @param {type:"slider", min:1, max:8, step:1}
aria_connections = 16 # @param {type:"slider", min:1, max:16, step:1}
aria_split = 16 # @param {type:"slider", min:1, max:16, step:1}
min_split_size = '1M' # @param ["1M", "2M", "4M", "8M", "16M"]
# @markdown ---
# @markdown ### Safety
skip_completed_files = True # @param {type:"boolean"}
fallback_to_wget = True # @param {type:"boolean"}

from nenen88 import download_many

def _request(value, target):
    value = str(value).strip()
    if not value or target is None:
        return ''
    parts = value.split()
    if len(parts) == 1:
        return f'{value} {target}'
    return f"{parts[0]} {target} {' '.join(parts[1:])}"

_queue = []
for _url in [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]:
    _queue.append(_request(_url, CKPT))
for _url in [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]:
    _queue.append(_request(_url, LORA))
_queue.append(_request(VAE_URL, VAE))

_download_results = download_many(
    _queue,
    max_workers=parallel_workers,
    parallel=(download_mode == 'parallel'),
    aria_connections=aria_connections,
    aria_split=aria_split,
    min_split_size=min_split_size,
    skip_completed=skip_completed_files,
    fallback_to_wget=fallback_to_wget,
)


## 🛠️ Extra Assets *(Optional)*


In [ ]:
# @title 🛠️ **Extra Assets** {"display-mode":"form"}
# @markdown ### 🔌 Extensions / Custom Nodes (git clone URL)
Extension_1 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
# @markdown ---
# @markdown ### 🖼️ Embeddings
Embedding_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### 🔬 Upscalers
Upscaler_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### ⚡ Download Mode
extra_download_mode = 'parallel' # @param ["parallel", "sequence"]
extra_parallel_workers = 6 # @param {type:"slider", min:1, max:8, step:1}
extra_aria_connections = 16 # @param {type:"slider", min:1, max:16, step:1}
extra_aria_split = 16 # @param {type:"slider", min:1, max:16, step:1}
extra_min_split_size = '1M' # @param ["1M", "2M", "4M", "8M", "16M"]
extra_skip_completed_files = True # @param {type:"boolean"}
extra_fallback_to_wget = True # @param {type:"boolean"}

import tempfile, os
from nenen88 import clone, download_many

_ext_urls = [u.strip() for u in [Extension_1, Extension_2, Extension_3] if u.strip()]
if _ext_urls:
    _tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False)
    _tmp.write('\n'.join(_ext_urls))
    _tmp.flush()
    _tmp.close()
    print('[INFO] Cloning extensions/custom nodes...')
    %cd -q $Extensions
    get_ipython().run_line_magic('clone', _tmp.name)
    os.unlink(_tmp.name)

_queue = []
for _url in [Embedding_1, Embedding_2]:
    if str(_url).strip():
        _queue.append(f'{str(_url).strip()} {Embeddings}')
for _url in [Upscaler_1, Upscaler_2]:
    if str(_url).strip():
        _queue.append(f'{str(_url).strip()} {Upscalers}')

if _queue:
    _download_results = download_many(
        _queue,
        max_workers=extra_parallel_workers,
        parallel=(extra_download_mode == 'parallel'),
        aria_connections=extra_aria_connections,
        aria_split=extra_aria_split,
        min_split_size=extra_min_split_size,
        skip_completed=extra_skip_completed_files,
        fallback_to_wget=extra_fallback_to_wget,
    )
else:
    print('[INFO] No extra asset URL provided.')


## ⚡ FLUX Models *(Optional)*


In [ ]:
# @title ⚡ **FLUX Model Downloader** {"display-mode":"form"}
FLUX_Variant = 'None' # @param ["None", "FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
FLUX_Unet = 'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-schnell-fp8.safetensors' # @param {type:"string"}
FLUX_Clip_L = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors' # @param {type:"string"}
FLUX_T5XXL = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors' # @param {type:"string"}
FLUX_VAE = 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors' # @param {type:"string"}
# @markdown ---
flux_download_mode = 'parallel' # @param ["parallel", "sequence"]
flux_parallel_workers = 3 # @param {type:"slider", min:1, max:8, step:1}
flux_aria_connections = 16 # @param {type:"slider", min:1, max:16, step:1}
flux_aria_split = 16 # @param {type:"slider", min:1, max:16, step:1}
flux_min_split_size = '1M' # @param ["1M", "2M", "4M", "8M", "16M"]
flux_skip_completed_files = True # @param {type:"boolean"}
flux_fallback_to_wget = True # @param {type:"boolean"}

from nenen88 import download_many

if FLUX_Variant == 'None':
    print('[INFO] FLUX_Variant is None; skipping.')
else:
    _unet = FLUX_Unet.replace('schnell', 'dev') if 'dev' in FLUX_Variant.lower() and 'schnell' in FLUX_Unet else FLUX_Unet
    _queue = []
    for _url, _target, _name in [
        (_unet, UNET, None),
        (FLUX_Clip_L, CLIP, None),
        (FLUX_T5XXL, CLIP, None),
        (FLUX_VAE, VAE, 'flux_ae.safetensors'),
    ]:
        if _url and _target is not None:
            _queue.append(f'{_url} {_target}' + (f' {_name}' if _name else ''))
    _download_results = download_many(
        _queue,
        max_workers=flux_parallel_workers,
        parallel=(flux_download_mode == 'parallel'),
        aria_connections=flux_aria_connections,
        aria_split=flux_aria_split,
        min_split_size=flux_min_split_size,
        skip_completed=flux_skip_completed_files,
        fallback_to_wget=flux_fallback_to_wget,
    )


## 🎛️ ControlNet *(Optional)*


In [ ]:
# @title 🎛️ **ControlNet Widget** {"display-mode":"form"}
ControlNet_Parallel_Download = True # @param {type:"boolean"}
ControlNet_Max_Workers = 6 # @param {type:"slider", min:1, max:8, step:1}

import os
os.environ['SEGSM_PARALLEL_DOWNLOAD'] = '1' if ControlNet_Parallel_Download else '0'
os.environ['SEGSM_MAX_WORKERS'] = str(ControlNet_Max_Workers)

%run $Controlnet_Widget


## 💨 Temporary Models


In [ ]:
# @title 💨 **Temporary Model Downloader** {"display-mode":"form"}
TMP_Checkpoint_1 = '' # @param {type:"string", placeholder:"URL [filename] or leave empty"}
TMP_Checkpoint_2 = '' # @param {type:"string", placeholder:"URL [filename] or leave empty"}
TMP_Lora_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
TMP_Lora_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
tmp_download_mode = 'parallel' # @param ["parallel", "sequence"]
tmp_parallel_workers = 4 # @param {type:"slider", min:1, max:8, step:1}
tmp_aria_connections = 16 # @param {type:"slider", min:1, max:16, step:1}
tmp_aria_split = 16 # @param {type:"slider", min:1, max:16, step:1}
tmp_min_split_size = '1M' # @param ["1M", "2M", "4M", "8M", "16M"]
tmp_skip_completed_files = True # @param {type:"boolean"}
tmp_fallback_to_wget = True # @param {type:"boolean"}

from nenen88 import download_many

def _tmp_request(raw, target):
    raw = str(raw).strip()
    if not raw:
        return ''
    parts = raw.split(None, 1)
    return f'{parts[0]} {target}' + (f' {parts[1]}' if len(parts) > 1 else '')

_queue = [
    _tmp_request(TMP_Checkpoint_1, TMP_CKPT),
    _tmp_request(TMP_Checkpoint_2, TMP_CKPT),
    _tmp_request(TMP_Lora_1, TMP_LORA),
    _tmp_request(TMP_Lora_2, TMP_LORA),
]
_download_results = download_many(
    _queue,
    max_workers=tmp_parallel_workers,
    parallel=(tmp_download_mode == 'parallel'),
    aria_connections=tmp_aria_connections,
    aria_split=tmp_aria_split,
    min_split_size=tmp_min_split_size,
    skip_completed=tmp_skip_completed_files,
    fallback_to_wget=tmp_fallback_to_wget,
)


# 🚀 Launch


In [ ]:
# @title 🚀 Launcher WebUI {display-mode:"form"}
# @markdown Pilih software dan masukkan token tunnel jika diperlukan.
Software = 'Forge' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Ngrok_Token = '' # @param {type:"string", placeholder:"Optional NGROK token"}
Zrok_Token = '' # @param {type:"string", placeholder:"Optional ZROK token"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}

recommended_args = {
    'A1111': '--xformers',
    'Forge': '--disable-xformers --opt-sdp-attention --cuda-stream',
    'ReForge': '--xformers --cuda-stream',
    'ReForge-old': '--xformers --cuda-stream',
    'Forge-Classic': '--xformers --cuda-stream --persistent-patches',
    'Forge-Neo': '--xformers --cuda-malloc --cuda-stream',
    'ComfyUI': '--dont-print-server --use-pytorch-cross-attention',
    'SwarmUI': '--launch_mode none',
}

selected_args = recommended_args.get(Software, '')
if Skip_ComfyUI_Check:
    selected_args += ' --skip-comfyui-check'
if Ngrok_Token.strip():
    selected_args += f' --N={Ngrok_Token.strip()}'
if Zrok_Token.strip():
    selected_args += f' --Z={Zrok_Token.strip()}'

print(f'[INFO] Launching {Software}')
%cd -q $WebUI
# Equivalent form command: %run segsmaker.py
get_ipython().run_line_magic('run', f'segsmaker.py {selected_args}')
